# K-means en 2D: caso multiclase con cinco grupos

Este notebook extiende el ejemplo bidimensional a cinco grupos. El objetivo es observar el comportamiento de K-means cuando existen varias nubes de puntos y se solicita al algoritmo encontrar cinco centroides.

## 1. De dos clusters a cinco clusters

Cuando se pasa de $K=2$ a $K=5$, el algoritmo conserva la misma lógica:

1. ubicar centroides iniciales;
2. asignar cada punto al centroide más cercano;
3. recalcular los centroides;
4. repetir hasta estabilizar las asignaciones.

La diferencia es que ahora el espacio se divide en cinco regiones. Cada región queda asociada a un centroide.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from itertools import permutations
from sklearn.cluster import KMeans
from sklearn.metrics import accuracy_score, adjusted_rand_score, confusion_matrix

RANDOM_STATE = 7
rng = np.random.default_rng(RANDOM_STATE)

plt.rcParams["figure.figsize"] = (7, 5)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.25

In [ ]:
def mejor_acierto_por_permutacion(y_true, y_cluster):
    """Calcula el mejor acierto posible cuando los nombres de los clusters son arbitrarios.

    K-means no sabe que una clase debe llamarse 0, 1, 2, etc. Por eso, antes de
    calcular accuracy, se prueban todas las correspondencias posibles entre
    clusters encontrados y etiquetas verdaderas.
    """
    y_true = np.asarray(y_true)
    y_cluster = np.asarray(y_cluster)

    etiquetas_verdaderas = np.unique(y_true)
    etiquetas_cluster = np.unique(y_cluster)

    if len(etiquetas_verdaderas) != len(etiquetas_cluster):
        raise ValueError("El número de etiquetas verdaderas y clusters debe coincidir.")

    mejor_accuracy = -1.0
    mejor_mapeo = None
    mejor_y_pred = None

    for perm in permutations(etiquetas_verdaderas):
        mapeo = {cluster: etiqueta for cluster, etiqueta in zip(etiquetas_cluster, perm)}
        y_pred = np.array([mapeo[c] for c in y_cluster])
        acc = accuracy_score(y_true, y_pred)

        if acc > mejor_accuracy:
            mejor_accuracy = acc
            mejor_mapeo = mapeo
            mejor_y_pred = y_pred

    return mejor_accuracy, mejor_y_pred, mejor_mapeo

## 2. Generación de datos para cinco clases

Se definen cinco medias en el plano. Para simplificar, todas las clases usan la misma matriz de covarianza identidad:

$$
\Sigma = I_2
$$

Esto genera nubes aproximadamente circulares y facilita la interpretación visual.

In [ ]:
mu = np.array([
    [2, 3],    # Clase 0
    [6, 8],    # Clase 1
    [10, 3],   # Clase 2
    [2, 10],   # Clase 3
    [8, 10],   # Clase 4
])

n_classes = mu.shape[0]
n_points = 100
sigma = np.eye(2)

data_parts = []
true_labels_parts = []

for i in range(n_classes):
    class_data = rng.multivariate_normal(mu[i], sigma, n_points)
    data_parts.append(class_data)
    true_labels_parts.append(np.full(n_points, i, dtype=int))

data = np.vstack(data_parts)
true_labels = np.concatenate(true_labels_parts)

df = pd.DataFrame(data, columns=["coordenada_x", "coordenada_y"])
df["etiqueta_verdadera"] = true_labels
df.head()

## 3. Visualización de los datos sin etiquetas

La siguiente gráfica representa el escenario que recibe K-means: una nube de puntos en dos dimensiones sin información explícita de clase.

In [ ]:
plt.figure()
plt.scatter(data[:, 0], data[:, 1], s=35, alpha=0.8)
plt.title("Datos 2D sin etiquetar: cinco grupos")
plt.xlabel("Coordenada X")
plt.ylabel("Coordenada Y")
plt.show()

## 4. Visualización de las clases reales

Esta gráfica muestra las clases reales usadas para generar los datos. Se incluye para comparar visualmente el resultado del algoritmo, pero estas etiquetas no se entregan a K-means durante el entrenamiento.

In [ ]:
plt.figure()
plt.scatter(data[:, 0], data[:, 1], c=true_labels, cmap="tab10", s=35, alpha=0.85)
plt.title("Etiquetas verdaderas de las cinco clases")
plt.xlabel("Coordenada X")
plt.ylabel("Coordenada Y")
plt.show()

## 5. Aplicación de K-means con cinco clusters

Como se desea identificar cinco grupos, se define:

$$
K = 5
$$

In [ ]:
num_clusters = 5

kmeans = KMeans(n_clusters=num_clusters, random_state=RANDOM_STATE, n_init=10)
cluster_idx = kmeans.fit_predict(data)
cluster_centers = kmeans.cluster_centers_

pd.DataFrame(cluster_centers, columns=["centroide_x", "centroide_y"])

## 6. Resultado de K-means

La gráfica muestra los clusters encontrados y sus centroides. Si las nubes están bien separadas, cada centroide debería ubicarse cerca del centro de una nube de puntos.

In [ ]:
plt.figure()
plt.scatter(data[:, 0], data[:, 1], c=cluster_idx, cmap="tab10", s=35, alpha=0.85)
plt.scatter(cluster_centers[:, 0], cluster_centers[:, 1], marker="*", s=320, c="black", edgecolor="white", label="Centroides")
plt.title("Agrupamiento K-means con cinco clusters")
plt.xlabel("Coordenada X")
plt.ylabel("Coordenada Y")
plt.legend()
plt.show()

## 7. Comparación cuantitativa con etiquetas verdaderas

Como en este ejemplo las etiquetas verdaderas sí están disponibles, se puede medir la similitud entre los grupos generados y las clases reales. Primero se corrige la arbitrariedad de nombres de los clusters mediante permutaciones.

In [ ]:
accuracy, predicted_labels, mapping = mejor_acierto_por_permutacion(true_labels, cluster_idx)
ari = adjusted_rand_score(true_labels, cluster_idx)

print(f"Mejor correspondencia cluster → etiqueta verdadera: {mapping}")
print(f"Accuracy corregido por permutación: {accuracy:.4f}")
print(f"Adjusted Rand Index: {ari:.4f}")

## 8. Matriz de confusión corregida

La matriz permite identificar cuáles clases se confunden entre sí después de asignar los nombres de cluster más convenientes.

In [ ]:
cm = confusion_matrix(true_labels, predicted_labels)
cm_df = pd.DataFrame(
    cm,
    index=[f"Clase real {i}" for i in range(n_classes)],
    columns=[f"Predicha {i}" for i in range(n_classes)]
)
cm_df

## 9. Actividad propuesta

Acerque algunas medias en la variable `mu`. Por ejemplo, acerque la clase 1 y la clase 4. Luego vuelva a ejecutar el notebook y analice:

1. cómo cambian los centroides;
2. qué clases aparecen más confundidas en la matriz de confusión;
3. si el valor de `Adjusted Rand Index` disminuye.